In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- quant_backtester_calculate_returns_migration ---
FIX_QUANT_BACKTESTER_CALCULATE_RETURNS_MIGRATION_SIGNALS = pd.DataFrame({"positions": [0, 1, 1, -1, 0]}, index=pd.date_range("2023-01-01", periods=5))
FIX_QUANT_BACKTESTER_CALCULATE_RETURNS_MIGRATION_SIGNALS_PL = pl.DataFrame({"positions": [0, 1, 1, -1, 0]})

# --- quant_backtester_performance_metrics_migration ---

print("✅ Fixtures loaded")
SELF_DATA_PAIRS_PD = pd.DataFrame({"Close_1":[100.0,101.0,102.0,101.5,103.0],"Close_2":[50.0,50.5,51.0,50.8,51.5]}, index=pd.date_range("2023-01-01",periods=5))
SELF_DATA_PAIRS_PL = pl.from_pandas(SELF_DATA_PAIRS_PD.reset_index(drop=True))
SELF_DATA_SINGLE_PD = pd.DataFrame({"Close":[100.0,101.0,100.5,102.0,102.0]}, index=pd.date_range("2023-01-01",periods=5))
SELF_DATA_SINGLE_PL = pl.from_pandas(SELF_DATA_SINGLE_PD.reset_index(drop=True))
SELF_RESULTS_PD = pd.DataFrame({
    "return": [0.01,0.02,-0.01,0.015],
    "strategy_returns":[0.01,0.02,-0.01,0.015],
    "cumulative_returns":[1.01,1.03,1.02,1.035],
    "equity_curve": [10100.,10300.,10200.,10350.],
}, index=pd.date_range("2023-01-01", periods=4))
SELF_RESULTS_PL = pl.from_pandas(SELF_RESULTS_PD.reset_index(drop=True))
self = SimpleNamespace(data=SELF_DATA_PAIRS_PD, initial_capital=10000.0, results=SELF_RESULTS_PD)


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_quant_backtester_calculate_returns_migration(signals):
    portfolio = pd.DataFrame(index=signals.index)
    portfolio["positions"] = signals["positions"]

    # Pairs trading
    if "Close_1" in self.data.columns and "Close_2" in self.data.columns:
        portfolio["asset_returns"] = (
            self.data["Close_1"].pct_change() - self.data["Close_2"].pct_change()
        )
    # Single asset trading
    elif "Close" in self.data.columns:
        portfolio["asset_returns"] = self.data["Close"].pct_change()
    else:
        raise ValueError("Data does not contain required 'Close' columns")

    portfolio["strategy_returns"] = (
        portfolio["positions"].shift(1) * portfolio["asset_returns"]
    )
    # Handle potential NaN or inf values
    portfolio["strategy_returns"] = (
        portfolio["strategy_returns"].replace([np.inf, -np.inf], np.nan).fillna(0)
    )
    portfolio["cumulative_returns"] = (1 + portfolio["strategy_returns"]).cumprod()
    portfolio["equity_curve"] = (
        self.initial_capital * portfolio["cumulative_returns"]
    )
    return portfolio

def before_quant_backtester_performance_metrics_migration():
    total_return = self.results["cumulative_returns"].iloc[-1] - 1

    returns_mean = self.results["strategy_returns"].mean()
    returns_std = self.results["strategy_returns"].std()
    if returns_std != 0 and not np.isnan(returns_std):
        sharpe_ratio = np.sqrt(252) * returns_mean / returns_std
    else:
        sharpe_ratio = np.nan

    drawdowns = (
        self.results["equity_curve"] / self.results["equity_curve"].cummax() - 1
    )
    max_drawdown = drawdowns.min()
    return max_drawdown

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_quant_backtester_calculate_returns_migration(signals):

    portfolio = pl.DataFrame({"positions": signals["positions"]})

    # Pairs trading
    if "Close_1" in self.data.columns and "Close_2" in self.data.columns:
        portfolio = portfolio.with_columns(
            (
                self.data["Close_1"].pct_change() - self.data["Close_2"].pct_change()
            ).alias("asset_returns")
        )
    # Single asset trading
    elif "Close" in self.data.columns:
        portfolio = portfolio.with_columns(
            self.data["Close"].pct_change().alias("asset_returns")
        )
    else:
        raise ValueError("Data does not contain required 'Close' columns")

    portfolio = portfolio.with_columns(
        (pl.col("positions").shift(1) * pl.col("asset_returns")).alias("strategy_returns")
    )

    # Handle potential NaN or inf values
    portfolio = portfolio.with_columns(
        pl.when(pl.col("strategy_returns").is_finite())
        .then(pl.col("strategy_returns"))
        .otherwise(0)
        .alias("strategy_returns")
    )

    portfolio = portfolio.with_columns(
        (1 + pl.col("strategy_returns")).cum_prod().alias("cumulative_returns"),
        (self.initial_capital * pl.col("cumulative_returns")).alias("equity_curve"),
    )
    return portfolio

def gen_quant_backtester_performance_metrics_migration():
    import numpy as np

    total_return = self.results.get_column("cumulative_returns")[-1] - 1

    returns_mean = self.results.get_column("strategy_returns").mean()
    returns_std = self.results.get_column("strategy_returns").std()
    if returns_std is not None and returns_std != 0 and not np.isnan(returns_std):
        sharpe_ratio = np.sqrt(252) * returns_mean / returns_std
    else:
        sharpe_ratio = np.nan

    drawdowns = (
        self.results.get_column("equity_curve") / self.results.get_column("equity_curve").cum_max() - 1
    )
    max_drawdown = drawdowns.min()
    return max_drawdown

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: quant_backtester_performance_metrics_migration ===

try:
    _old_results = self.results
    self.results = SELF_RESULTS_PL
    _r = gen_quant_backtester_performance_metrics_migration()
    print("✅ L1 smoke gen_quant_backtester_performance_metrics_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_quant_backtester_performance_metrics_migration: {type(_e).__name__}: {_e}")
finally:
    self.results = _old_results

try:
    _old_results = self.results
    self.results = SELF_RESULTS_PD
    _rb = before_quant_backtester_performance_metrics_migration()
    print("✅ L1 smoke before_quant_backtester_performance_metrics_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_quant_backtester_performance_metrics_migration: {type(_e).__name__}: {_e}")
finally:
    self.results = _old_results

try:
    _old_results = self.results
    self.results = SELF_RESULTS_PD
    _rb = before_quant_backtester_performance_metrics_migration()
    self.results = SELF_RESULTS_PL
    _rg = gen_quant_backtester_performance_metrics_migration()
    if np.isclose(_rb, _rg):
        print("✅ L2 equivalence quant_backtester_performance_metrics_migration: MATCH")
    else:
        print(f"❌ L2 equivalence quant_backtester_performance_metrics_migration: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence quant_backtester_performance_metrics_migration: setup error — {type(_e).__name__}: {_e}")
finally:
    self.results = _old_results

# L3 branch – performance metrics with variable and zero-variance returns
try:
    _old_results = self.results
    self.results = SELF_RESULTS_PD
    _before_edge = before_quant_backtester_performance_metrics_migration()
    self.results = SELF_RESULTS_PL
    _gen_edge = gen_quant_backtester_performance_metrics_migration()
    if np.isclose(_before_edge, _gen_edge):
        print("✅ L3 branch quant_backtester_performance_metrics_migration drawdown: MATCH")
    else:
        print(f"❌ L3 branch quant_backtester_performance_metrics_migration drawdown: MISMATCH — before={_before_edge}, gen={_gen_edge}")
    _flat_pd = pd.DataFrame({"strategy_returns": [0.0, 0.0, 0.0], "cumulative_returns": [1.0, 1.0, 1.0], "equity_curve": [100.0, 100.0, 100.0]})
    _flat_pl = pl.from_pandas(_flat_pd)
    self.results = _flat_pd
    _before_flat = before_quant_backtester_performance_metrics_migration()
    self.results = _flat_pl
    _gen_flat = gen_quant_backtester_performance_metrics_migration()
    if np.isclose(_before_flat, _gen_flat):
        print("✅ L3 branch quant_backtester_performance_metrics_migration zero variance: MATCH")
    else:
        print(f"❌ L3 branch quant_backtester_performance_metrics_migration zero variance: MISMATCH — before={_before_flat}, gen={_gen_flat}")
finally:
    self.results = _old_results


✅ L1 smoke gen_quant_backtester_performance_metrics_migration: OK, type= float
✅ L1 smoke before_quant_backtester_performance_metrics_migration: OK
✅ L2 equivalence quant_backtester_performance_metrics_migration: MATCH
✅ L3 branch quant_backtester_performance_metrics_migration drawdown: MATCH
✅ L3 branch quant_backtester_performance_metrics_migration zero variance: MATCH
